In [ ]:
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok nltk streamlit-option-menu plotly textstat PyPDF2 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 43.4 MB/s eta 0:00:00


In [ ]:
%%writefile db.py
import sqlite3
import bcrypt
import datetime
import time
import os

DB_NAME = "users.db"

MAX_ATTEMPTS = 3
LOCKOUT_SECONDS = 120

ADMIN_EMAIL = os.getenv("ADMIN_EMAIL", "bhavana1905.ai@gmail.com")
ADMIN_PASSWORD = os.getenv("ADMIN_PASSWORD", "Bhavana@123")



# ==========================
# INIT
# ==========================

def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("""
    CREATE TABLE IF NOT EXISTS users (
        email TEXT PRIMARY KEY,
        username TEXT UNIQUE,
        password BLOB,
        security_question TEXT,
        security_answer TEXT,
        created_at TEXT
    )
    """)

    c.execute("""
    CREATE TABLE IF NOT EXISTS password_history (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        email TEXT,
        password BLOB,
        set_at TEXT
    )
    """)

    c.execute("""
    CREATE TABLE IF NOT EXISTS login_attempts (
        email TEXT PRIMARY KEY,
        attempts INTEGER DEFAULT 0,
        last_attempt REAL
    )
    """)

    conn.commit()
    conn.close()

    init_admin()


def init_admin():
    if ADMIN_PASSWORD and not check_user_exists(ADMIN_EMAIL):
        register_user(
            ADMIN_EMAIL,
            "BhavanaAdmin",
            ADMIN_PASSWORD,
            "What is your pet name?",
            "admin"
        )


def _get_timestamp():
    return datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


# ==========================
# USER MANAGEMENT
# ==========================

def register_user(email, username, password, security_question, security_answer):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    try:
        hashed = bcrypt.hashpw(password.encode(), bcrypt.gensalt())
        now = _get_timestamp()

        c.execute("""
        INSERT INTO users VALUES (?, ?, ?, ?, ?, ?)
        """, (email, username, hashed, security_question, security_answer.lower(), now))

        c.execute("""
        INSERT INTO password_history (email, password, set_at)
        VALUES (?, ?, ?)
        """, (email, hashed, now))

        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()


def authenticate_user(email, password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password FROM users WHERE email=?", (email,))
    data = c.fetchone()
    conn.close()

    if data and bcrypt.checkpw(password.encode(), data[0]):
        reset_attempts(email)
        return True

    record_failed_attempt(email)
    return False


def check_user_exists(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT 1 FROM users WHERE email=?", (email,))
    exists = c.fetchone() is not None
    conn.close()
    return exists

def verify_security_answer(email, answer):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT security_answer FROM users WHERE email=?", (email,))
    row = c.fetchone()
    conn.close()

    if row and row[0] == answer.lower():
        return True
    return False


# ==========================
# PASSWORD MANAGEMENT
# ==========================

def check_password_reused(email, new_password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password FROM password_history WHERE email=?", (email,))
    history = c.fetchall()
    conn.close()

    for (stored_hash,) in history:
        if bcrypt.checkpw(new_password.encode(), stored_hash):
            return True
    return False

def check_is_old_password(email, password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("""
        SELECT password, set_at
        FROM password_history
        WHERE email=?
        ORDER BY set_at DESC
    """, (email,))

    rows = c.fetchall()
    conn.close()

    for stored_hash, set_at in rows:
        if bcrypt.checkpw(password.encode(), stored_hash):
            return set_at

    return None


def update_password(email, new_password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    hashed = bcrypt.hashpw(new_password.encode(), bcrypt.gensalt())
    now = _get_timestamp()

    c.execute("UPDATE users SET password=? WHERE email=?", (hashed, email))
    c.execute("INSERT INTO password_history (email, password, set_at) VALUES (?, ?, ?)",
              (email, hashed, now))

    conn.commit()
    conn.close()


# ==========================
# RATE LIMITING
# ==========================

def record_failed_attempt(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    now = time.time()

    c.execute("SELECT attempts, last_attempt FROM login_attempts WHERE email=?", (email,))
    row = c.fetchone()

    if row:
        attempts, last = row
        if now - last > LOCKOUT_SECONDS:
            attempts = 1
        else:
            attempts += 1
        c.execute("UPDATE login_attempts SET attempts=?, last_attempt=? WHERE email=?",
                  (attempts, now, email))
    else:
        c.execute("INSERT INTO login_attempts VALUES (?,1,?)", (email, now))

    conn.commit()
    conn.close()


def reset_attempts(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM login_attempts WHERE email=?", (email,))
    conn.commit()
    conn.close()


def is_rate_limited(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT attempts, last_attempt FROM login_attempts WHERE email=?", (email,))
    row = c.fetchone()
    conn.close()

    if row:
        attempts, last = row
        if attempts >= MAX_ATTEMPTS and (time.time() - last) < LOCKOUT_SECONDS:
            return True, LOCKOUT_SECONDS - (time.time() - last)

    return False, 0


# ==========================
# ADMIN
# ==========================

def get_all_users():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT email, username, created_at FROM users ORDER BY created_at DESC")
    data = c.fetchall()
    conn.close()
    return data


def delete_user(email):
    if email == ADMIN_EMAIL:
        return False

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM users WHERE email=?", (email,))
    c.execute("DELETE FROM password_history WHERE email=?", (email,))
    c.execute("DELETE FROM login_attempts WHERE email=?", (email,))
    conn.commit()
    conn.close()
    return True


Writing db.py


In [ ]:
%%writefile readability.py
import textstat

class ReadabilityAnalyzer:
    def __init__(self, text):
        self.text = text
        self.num_sentences = textstat.sentence_count(text)
        self.num_words = textstat.lexicon_count(text, removepunct=True)
        self.num_syllables = textstat.syllable_count(text)
        self.complex_words = textstat.difficult_words(text)
        self.char_count = textstat.char_count(text)

    def get_all_metrics(self):
        return {
            "Flesch Reading Ease": textstat.flesch_reading_ease(self.text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(self.text),
            "SMOG Index": textstat.smog_index(self.text),
            "Gunning Fog": textstat.gunning_fog(self.text),
            "Coleman-Liau": textstat.coleman_liau_index(self.text)
        }


Writing readability.py


In [ ]:
%%writefile app.py
import sqlite3
import streamlit as st
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import secrets
import bcrypt
import jwt
import datetime
import time
import os
import re
import hmac
import hashlib
import struct
import db
from streamlit_option_menu import option_menu
import plotly.graph_objects as go
import PyPDF2

# --- Configuration ---
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")
SECRET_KEY = os.getenv("JWT_SECRET", "JWT_secret_key")
ALGORITHM = "HS256"
OTP_EXPIRY_MINUTES = 10
ACCESS_TOKEN_EXPIRE_MINUTES = 30
MAX_LOGIN_ATTEMPTS = 3
LOCKOUT_SECONDS = 300

# --- Database Initialization ---
if 'db_initialized' not in st.session_state:
    db.init_db()
    st.session_state['db_initialized'] = True

# --- UI Theme (Neon Style) ---
st.set_page_config(page_title="TextMorph – Advanced Text Summarization and Paraphrasing", layout="wide")

def apply_clean_theme():
    st.markdown("""
    <style>

/* ========================= */
/* Gradient Background */
/* ========================= */

.stApp {
    background: linear-gradient(135deg,#ff6ec7,#845ec2,#2c73d2,#00c9a7,#f9f871);
    background-size: 400% 400%;
    animation: gradientMove 12s ease infinite;
    font-family: "Segoe UI", sans-serif;
}

@keyframes gradientMove {
    0%{background-position:0% 50%;}
    50%{background-position:100% 50%;}
    100%{background-position:0% 50%;}
}

/* Remove black header background */

header[data-testid="stHeader"]{
    background: transparent !important;
}

/* remove border shadow */

header[data-testid="stHeader"]::before{
    box-shadow: none !important;
}

/* keep menu visible */

button[kind="header"]{
    color: white !important;
}
/* REMOVE STREAMLIT DEFAULT INPUT BORDER */

div[data-baseweb="input"]{
    border: none !important;
    box-shadow: none !important;
    outline: none !important;
    background: transparent !important;
}

/* also remove extra wrapper border */

div[data-baseweb="base-input"]{
    border: none !important;
    box-shadow: none !important;
}

/* GLASS INPUT STYLE */

.stTextInput input{
    background: rgba(255,255,255,0.15) !important;
    backdrop-filter: blur(20px);

    border-radius: 50px !important;
    border: 1px solid rgba(255,255,255,0.4) !important;

    color: white !important;
    padding: 14px 18px;
}
/* ========================= */
/* FIX INPUT FIELD WRAPPER */
/* ========================= */

.stTextInput > div {
    background: transparent !important;
}

.stTextInput > div > div {
    background: rgba(255,255,255,0.15) !important;
    backdrop-filter: blur(20px);
    border-radius: 50px;
    border: 1px solid rgba(255,255,255,0.5);
}

/* ========================= */
/* INPUT TEXT */
/* ========================= */

.stTextInput input {
    background: transparent !important;
    color: white !important;
    border: none !important;
    padding: 14px 18px;
    font-size: 16px;
}

/* remove default focus box */
.stTextInput input:focus {
    outline: none !important;
    box-shadow: none !important;
}

/* placeholder */

.stTextInput input::placeholder {
    color: rgba(255,255,255,0.7);
}

/* ========================= */
/* PASSWORD FIELD */
/* ========================= */

.stTextInput input[type="password"] {
    background: transparent !important;
}

/* ========================= */
/* BUTTON STYLE */
/* ========================= */

.stButton > button,
.stFormSubmitButton > button {

    background: linear-gradient(135deg,#00c9a7,#845ec2) !important;
    color: white !important;

    border-radius: 12px !important;
    border: none !important;

    padding: 8px 20px !important;

    font-weight: 600;

    transition: all 0.3s ease;
}

/* hover */

.stButton > button:hover,
.stFormSubmitButton > button:hover {

    background: linear-gradient(135deg,#2c73d2,#ff6ec7) !important;

    transform: translateY(-2px);

    box-shadow: 0 4px 12px rgba(0,0,0,0.3);
}
/* ========================= */
/* SIDEBAR GLASS PANEL */
/* ========================= */

section[data-testid="stSidebar"]{

    background: rgba(255,255,255,0.08) !important;

    backdrop-filter: blur(20px);
    -webkit-backdrop-filter: blur(20px);

    border-right: 1px solid rgba(255,255,255,0.15);

    box-shadow: 4px 0 20px rgba(0,0,0,0.2);
}

/* remove white inner wrapper */

section[data-testid="stSidebar"] > div{
    background: transparent !important;
}

/* sidebar text */

section[data-testid="stSidebar"] *{
    color: white !important;
}

/* sidebar divider */

section[data-testid="stSidebar"] hr{
    border-color: rgba(255,255,255,0.15);
}
/* ========================= */
/* OPTION MENU GLASS */
/* ========================= */

[data-testid="stSidebar"] .nav{
    background: rgba(255,255,255,0.10) !important;
    backdrop-filter: blur(20px);
    border-radius: 18px;
    border: 1px solid rgba(255,255,255,0.15);
}

/* ============================= */
/* REMOVE BLACK FOOTER BAR */
/* ============================= */

footer {visibility: hidden;}
.stApp footer {visibility: hidden;}
section[data-testid="stFooter"] {display: none !important;}

/* Remove bottom black container */
[data-testid="stBottomBlockContainer"]{
    background: transparent !important;
}

/* ============================= */
/* REMOVE DARK BACKGROUND OF TAB */
/* ============================= */

div[data-testid="stVerticalBlock"] > div:has(.stChatInput){
    background: transparent !important;
}

/* Chat input area transparent */
.stChatInputContainer{
    background: transparent !important;
}

/* Remove dark block behind chat area */
.block-container{
    background: transparent !important;
}

/* ============================= */
/* OPTIONAL: MAKE INPUT FLOATING */
/* ============================= */

.stChatInputContainer{
    position: fixed;
    bottom: 30px;
    left: 50%;
    transform: translateX(-50%);
    width: 60%;
    background: rgba(255,255,255,0.08);
    backdrop-filter: blur(12px);
    border-radius: 25px;
    padding: 8px;
}


</style>
""", unsafe_allow_html=True)

apply_clean_theme()

# --- Helpers ---
def get_relative_time(date_str):
    if not date_str: return "some time ago"
    try:
        past = datetime.datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")
        diff = datetime.datetime.utcnow() - past
        days = diff.days
        seconds = diff.seconds
        if days > 365: return f"{days // 365} years ago"
        elif days > 30: return f"{days // 30} months ago"
        elif days > 0: return f"{days} days ago"
        elif seconds > 3600: return f"{seconds // 3600} hours ago"
        elif seconds > 60: return f"{seconds // 60} minutes ago"
        else: return "just now"
    except: return date_str


# ==============================
# JWT
# ==============================

def create_access_token(data: dict, expires_minutes=ACCESS_TOKEN_EXPIRE_MINUTES):
    to_encode = data.copy()
    expire = datetime.datetime.utcnow() + datetime.timedelta(minutes=expires_minutes)
    to_encode.update({"exp": expire})
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)

def verify_token(token):
    try:
        return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
    except:
        return None

def is_valid_email(email):
    return re.match(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$", email) is not None

def check_password_strength(password):
    has_upper = bool(re.search(r"[A-Z]", password))
    has_lower = bool(re.search(r"[a-z]", password))
    has_digit = bool(re.search(r"\d", password))
    has_special = bool(re.search(r"[!@#$%^&*(),.?\":{}|<>]", password))
    has_space = bool(re.search(r"\s", password))

    if has_space: return "Weak", ["No spaces allowed"]
    is_alphanum = (has_upper or has_lower) and has_digit

    if len(password) >= 8 and is_alphanum: return "Strong", []
    if len(password) >= 6 and is_alphanum and has_special: return "Medium", ["Add 2 more chars for Strong"]
    if len(password) >= 1: return "Weak", ["Too short (aim for 8+)"]
    return "Weak", ["Enter password"]

# --- Security Logic ---
def generate_otp():
    secret = secrets.token_bytes(20)
    counter = int(time.time())
    msg = struct.pack(">Q", counter)
    hmac_hash = hmac.new(secret, msg, hashlib.sha1).digest()
    offset = hmac_hash[19] & 0xf
    code = ((hmac_hash[offset] & 0x7f) << 24 | (hmac_hash[offset + 1] & 0xff) << 16 | (hmac_hash[offset + 2] & 0xff) << 8 | (hmac_hash[offset + 3] & 0xff))
    otp = code % 1000000
    return f"{otp:06d}"

def create_otp_token(otp, email):
    """Creates a JWT containing the hashed OTP, bound to the user's email."""
    otp_hash = bcrypt.hashpw(otp.encode(), bcrypt.gensalt()).decode()
    payload = {
        'otp_hash': otp_hash, 'sub': email, 'type': 'password_reset',
        'iat': datetime.datetime.utcnow(),
        'exp': datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)
    }
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

def verify_otp_token(token, input_otp, email):
    """Verifies the token signature, expiration, email binding, and OTP hash."""
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=['HS256'])
        if payload.get('type') != 'password_reset': return False, "Invalid token type"
        if payload.get('sub') != email: return False, "Token does not belong to this user"
        if bcrypt.checkpw(input_otp.encode('utf-8'), payload['otp_hash'].encode('utf-8')):
            return True, "Valid OTP"
        return False, "Invalid OTP"
    except jwt.ExpiredSignatureError: return False, "OTP Expired"
    except jwt.InvalidTokenError: return False, "Invalid Token"

# --- Email Logic ---
def send_email(receiver_email, otp):

    subject = "🔐 Infosys LLM - Password Reset OTP"

    body = f"""
    <html>
    <body style="font-family:Arial;">

    <h2>🔐 Password Reset OTP</h2>

    <p>Use the OTP below to reset your password for <b>{receiver_email}</b>.</p>

    <h1 style="color:#4CAF50;">{otp}</h1>

    <p>Valid for <b>{OTP_EXPIRY_MINUTES}</b> minutes. Do not share this code.</p>

    <hr>

    <p style="font-size:12px;">© 2026 Secure Auth System</p>

    </body>
    </html>
    """

    msg = MIMEMultipart()
    msg["From"] = f"Infosys LLM <{EMAIL_ADDRESS}>"
    msg["To"] = receiver_email
    msg["Subject"] = subject

    msg.attach(MIMEText(body, "html"))

    try:
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()

        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)

        server.sendmail(EMAIL_ADDRESS, receiver_email, msg.as_string())

        server.quit()

        return True, "Email sent successfully!"

    except Exception as e:
        return False, str(e)

# --- Visualization Helper ---
def create_gauge(value, title, min_val=0, max_val=100, color="#00ffcc"):
    fig = go.Figure(go.Indicator(
        mode = "gauge+number",
        value = value,
        title = {'text': title, 'font': {'color': color, 'size': 14}},
        number = {'font': {'color': color, 'size': 20}},
        gauge = {
            'axis': {'range': [min_val, max_val], 'tickwidth': 1, 'tickcolor': color},
            'bar': {'color': color},
            'bgcolor': "#1f2937",
            'borderwidth': 2,
            'bordercolor': "#374151",
            'steps': [
                {'range': [min_val, max_val], 'color': "#0e1117"}
            ],
        }
    ))
    fig.update_layout(paper_bgcolor="#0e1117", font={'color': "#ffffff", 'family': "Courier New"}, height=250, margin=dict(l=10, r=10, t=40, b=10))
    return fig

# --- Navigation & Routing ---
if 'user' not in st.session_state: st.session_state['user'] = None
if 'page' not in st.session_state: st.session_state['page'] = 'login'

def switch_page(page):
    st.session_state['page'] = page
    st.rerun()

def logout():
    st.session_state['user'] = None
    st.session_state['page'] = 'login'
    st.rerun()

# ========================================
# --- PAGES ---
# ========================================

def login_page():
    st.title("Login")
    st.markdown("### Secure Login")

    with st.form("login_form"):
        email = st.text_input("Email *")
        password = st.text_input("Password *", type='password')
        submit = st.form_submit_button("Login")

        if submit:
            is_locked, wait_time = db.is_rate_limited(email)
            if is_locked:
                st.error(f"⛔ Account Locked! Too many failed attempts. Try again in {int(wait_time)}s.")
            elif not email or not password:
                st.error("Please fill in all mandatory fields (*).")
            elif db.authenticate_user(email, password):
                st.session_state['user'] = email
                st.balloons()
                st.success(f"Welcome to Infosys LLM, {email}!")
                time.sleep(1)
                st.rerun()
            else:
                st.error("Invalid email or password.")
                old_dt = db.check_is_old_password(email, password)
                if old_dt:
                    st.warning(f"⚠️ You entered an old password from {get_relative_time(old_dt)}. Please use your latest password.")

    st.markdown("---")
    c1, c2 = st.columns(2)
    if c1.button("Create Account"): switch_page("register")
    if c2.button("Forgot Password?"): switch_page("forgot")

def register_page():
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 2, 1])

    with col2:
        st.title("Create Account")

        with st.form("signup_form"):
            username = st.text_input("Username *")
            email = st.text_input("Email Address *")
            password = st.text_input("Password *", type="password")
            confirm_password = st.text_input("Confirm Password *", type="password")

            security_question = st.selectbox(
                "Select Security Question",
                [
                    "What is your pet name?",
                    "What is your mother’s maiden name?",
                    "What is your favorite movie?"
                ]
            )

            security_answer = st.text_input("Security Answer *")

            submitted = st.form_submit_button("Sign Up")

            if submitted:
                errors = []

                # Username
                if not username:
                    errors.append("Username is mandatory.")

                # Email
                if not email:
                    errors.append("Email is mandatory.")
                elif not is_valid_email(email):
                    errors.append("Invalid Email format.")

                # Password
                if not password:
                    errors.append("Password is mandatory.")
                else:
                    strength, feedback = check_password_strength(password)
                    if strength == "Weak":
                        errors.append("Password too weak.")

                # Confirm password
                if password != confirm_password:
                    errors.append("Passwords do not match.")

                # Security answer
                if not security_answer:
                    errors.append("Security Answer is mandatory.")

                if errors:
                    for error in errors:
                        st.error(error)
                else:
                    success = db.register_user(
                        email,
                        username,
                        password,
                        security_question,
                        security_answer
                    )

                    if success:
                        st.success("Account created successfully! Please login.")
                        time.sleep(1)
                        switch_page("login")
                    else:
                        st.error("Email or Username already exists.")

        st.markdown("---")
        if st.button("Back to Login"):
            switch_page("login")

def forgot_page():
    st.title("🔐 Password Recovery")

    if 'stage' not in st.session_state:
        st.session_state['stage'] = 'email'

    # ======================
    # STEP 1 — EMAIL
    # ======================
    if st.session_state['stage'] == 'email':
        email = st.text_input("Enter Registered Email *")

        if st.button("Next"):
            if db.check_user_exists(email):
                st.session_state['reset_email'] = email
                st.session_state['stage'] = 'choose_method'
                st.rerun()
            else:
                st.error("Email not found.")

    # ======================
    # STEP 2 — CHOOSE METHOD
    # ======================
    elif st.session_state['stage'] == 'choose_method':
        st.subheader("Choose Recovery Method")

        col1, col2 = st.columns(2)

        if col1.button("Security Question"):
            st.session_state['stage'] = 'security'
            st.rerun()

        if col2.button("Email OTP"):
            st.session_state['stage'] = 'otp'
            st.rerun()

    # ======================
    # OPTION 1 — SECURITY QUESTION
    # ======================
    elif st.session_state['stage'] == 'security':
        conn = sqlite3.connect("users.db")
        c = conn.cursor()
        c.execute("SELECT security_question FROM users WHERE email=?",
                  (st.session_state['reset_email'],))
        row = c.fetchone()
        conn.close()

        if row:
            st.info(row[0])

            answer = st.text_input("Enter Answer *")

            if st.button("Verify Answer"):
                if db.verify_security_answer(
                        st.session_state['reset_email'], answer):
                    st.session_state['stage'] = 'reset'
                    st.success("Verified!")
                    st.rerun()
                else:
                    st.error("Incorrect answer.")

    # ======================
    # OPTION 2 — OTP
    # ======================
    elif st.session_state['stage'] == 'otp':
        if st.button("Send OTP"):
            otp = generate_otp()
            success, msg = send_email(
                st.session_state['reset_email'], otp
            )
            if success:
                st.session_state['token'] = create_otp_token(
                    otp,
                    st.session_state['reset_email']
                )
                st.session_state['stage'] = 'verify_otp'
                st.success("OTP Sent!")
                st.rerun()
            else:
                st.error(msg)

    elif st.session_state['stage'] == 'verify_otp':
        otp_input = st.text_input("Enter OTP *")

        if st.button("Verify OTP"):
            valid, msg = verify_otp_token(
                st.session_state['token'],
                otp_input,
                st.session_state['reset_email']
            )
            if valid:
                st.session_state['stage'] = 'reset'
                st.success("Verified!")
                st.rerun()
            else:
                st.error(msg)

    # ======================
    # FINAL STEP — RESET PASSWORD
    # ======================
    elif st.session_state['stage'] == 'reset':
      p1 = st.text_input("New Password *", type="password")
      p2 = st.text_input("Confirm Password *", type="password")

      if st.button("Update Password"):

        if not p1 or not p2:
            st.error("All password fields are mandatory.")
            return

        p1 = p1.strip()
        p2 = p2.strip()

        if " " in p1:
            st.error("Password cannot contain spaces.")
            return

        if p1 != p2:
            st.error("Passwords do not match.")
            return


        strength, feedback = check_password_strength(p1)
        if strength == "Weak":
            st.error(f"Password too weak: {', '.join(feedback)}")
            return


        if db.check_password_reused(
                st.session_state['reset_email'], p1):
            st.error("Cannot reuse old password.")
            return


        db.update_password(
            st.session_state['reset_email'], p1)

        st.success("Password Updated Successfully!")
        st.session_state.clear()
        time.sleep(1)
        switch_page("login")

def chat_page():
    if not st.session_state['user']: switch_page('login'); return
    st.title("🤖 Infosys LLM Chat")
    if "messages" not in st.session_state: st.session_state.messages = []
    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]): st.markdown(msg["content"])
    if prompt := st.chat_input("Ask me anything..."):
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"): st.markdown(prompt)
        with st.chat_message("assistant"):
            response = f"Simulated Response: {prompt} (Secure Mock)"
            st.markdown(response)
            st.session_state.messages.append({"role": "assistant", "content": response})

def readability_page():
    if not st.session_state['user']: switch_page('login'); return

    st.title("📖 Text Readability Analyzer")

    # Input Method: Text or File
    tab1, tab2 = st.tabs(["✍️ Input Text", "📂 Upload File (TXT/PDF)"])
    text_input = ""

    with tab1:
        raw_text = st.text_area("Enter text to analyze (min 50 chars):", height=200)
        if raw_text: text_input = raw_text

    with tab2:
        uploaded_file = st.file_uploader("Upload a file", type=["txt", "pdf"])
        if uploaded_file:
            try:
                if uploaded_file.type == "application/pdf":
                    reader = PyPDF2.PdfReader(uploaded_file)
                    text = ""
                    for page in reader.pages:
                        text += page.extract_text() + "\n"
                    text_input = text
                    st.info(f"✅ Loaded {len(reader.pages)} pages from PDF.")
                else:
                    text_input = uploaded_file.read().decode("utf-8")
                    st.info(f"✅ Loaded TXT file: {uploaded_file.name}")
            except Exception as e:
                st.error(f"Error reading file: {e}")

    if st.button("Analyze Readability", type="primary"):
        if len(text_input) < 50:
            st.error("Text is too short (min 50 chars). Please enter more text or upload a valid file.")
        else:
            import readability
            with st.spinner("Calculating advanced metrics..."):
                analyzer = readability.ReadabilityAnalyzer(text_input)
                score = analyzer.get_all_metrics()

            # --- Results Dashboard ---
            st.markdown("---")
            st.subheader("📊 Analysis Results")

            # 1. Overall Grade (Average)
            avg_grade = (score['Flesch-Kincaid Grade'] + score['Gunning Fog'] + score['SMOG Index'] + score['Coleman-Liau']) / 4

            # Determine Level
            if avg_grade <= 6: level, color = "Beginner (Elementary)", "#28a745"
            elif avg_grade <= 10: level, color = "Intermediate (Middle School)", "#17a2b8"
            elif avg_grade <= 14: level, color = "Advanced (High School/College)", "#ffc107"
            else: level, color = "Expert (Professional/Academic)", "#dc3545"

            st.markdown(f"""
            <div style="background-color: #1f2937; padding: 20px; border-radius: 10px; border-left: 5px solid {color}; text-align: center;">
                <h2 style="margin:0; color: {color} !important;">Overall Level: {level}</h2>
                <p style="margin:5px 0 0 0; color: #9ca3af;">Approximate Grade Level: {int(avg_grade)}</p>
            </div>
            """, unsafe_allow_html=True)

            st.markdown("### 📈 Detailed Metrics")

            # 2. Visual Gauges
            c1, c2, c3 = st.columns(3)
            with c1:
                st.plotly_chart(create_gauge(score["Flesch Reading Ease"], "Flesch Reading Ease", 0, 100, "#00ffcc"), use_container_width=True)
                with st.expander("ℹ️ About Flesch Ease"):
                    st.caption("0-100 Scale. Higher is easier. 60-70 is standard.")

            with c2:
                st.plotly_chart(create_gauge(score["Flesch-Kincaid Grade"], "Flesch-Kincaid Grade", 0, 20, "#ff00ff"), use_container_width=True)
                with st.expander("ℹ️ About Kincaid Grade"):
                    st.caption("US Grade Level. 8.0 means 8th grader can understand.")

            with c3:
                st.plotly_chart(create_gauge(score["SMOG Index"], "SMOG Index", 0, 20, "#ffff00"), use_container_width=True)
                with st.expander("ℹ️ About SMOG"):
                    st.caption("Commonly used for medical writing. Based on polysyllables.")

            c4, c5 = st.columns(2)
            with c4:
                st.plotly_chart(create_gauge(score["Gunning Fog"], "Gunning Fog", 0, 20, "#00ccff"), use_container_width=True)
                with st.expander("ℹ️ About Gunning Fog"):
                    st.caption("Based on sentence length and complex words.")

            with c5:
                st.plotly_chart(create_gauge(score["Coleman-Liau"], "Coleman-Liau", 0, 20, "#ff9900"), use_container_width=True)
                with st.expander("ℹ️ About Coleman-Liau"):
                    st.caption("Based on characters instead of syllables. Good for automated analysis.")

            # 3. Text Stats
            st.markdown("### 📝 Text Statistics")
            s1, s2, s3, s4, s5 = st.columns(5)
            s1.metric("Sentences", analyzer.num_sentences)
            s2.metric("Words", analyzer.num_words)
            s3.metric("Syllables", analyzer.num_syllables)
            s4.metric("Complex Words", analyzer.complex_words)
            s5.metric("Characters", analyzer.char_count)

def admin_page():
    if st.session_state['user'] != "admin@llm.com": st.error("Access Denied"); return
    st.title("🛡️ Admin Panel")
    users = db.get_all_users()

    st.metric("Total Users", len(users))
    st.markdown("---")

    c1, c2, c3 = st.columns([3, 2, 1])
    c1.markdown("**Email**"); c2.markdown("**Joined**"); c3.markdown("**Action**")
    st.markdown("---")
    for u_email, u_created in users:
        c1, c2, c3 = st.columns([3, 2, 1])
        c1.write(f"{u_email}"); c2.write(u_created)
        if u_email != "admin@llm.com":
            if c3.button("Delete", key=u_email, type="primary"):
                db.delete_user(u_email)
                st.warning(f"Deleted {u_email}");
                time.sleep(0.5);
                st.rerun()

# ========================================
# --- MAIN ROUTING WITH SIDEBAR ---
# ========================================

if st.session_state['user']:
    with st.sidebar:
        st.image("https://www.infosys.com/content/dam/infosys-web/en/about/springboard/images/infosys-springboard.png", width=150)
        st.markdown(f"**👤 {st.session_state['user']}**")
        st.markdown("---")

        opts = ["Chat", "Readability"]
        icons = ["chat-dots", "book"]
        if st.session_state['user'] == "admin@llm.com":
            opts.append("Admin"); icons.append("shield-lock")

        selected = option_menu("Infosys LLM", opts, icons=icons, menu_icon="cast", default_index=0,
            styles={
"container":{
    "background":"rgba(255,255,255,0.12)",
    "backdrop-filter":"blur(20px)",
    "border-radius":"18px",
    "padding":"10px",
    "border":"1px solid rgba(255,255,255,0.15)"
},

"icon":{
    "color":"white",
    "font-size":"18px"
},

"nav-link":{
    "color":"rgba(255,255,255,0.9)",
    "font-family":"Segoe UI",
    "font-size":"15px",
    "border-radius":"30px",
    "margin":"6px",
    "padding":"10px",
    "transition":"all 0.3s ease",
},

"nav-link-selected":{
    "background":"linear-gradient(135deg,#00c9a7,#2c73d2,#845ec2)",
    "color":"white",
    "font-weight":"600",
    "border-radius":"30px",
    "box-shadow":"0 0 14px rgba(0,201,167,0.6)"
}
})

        st.markdown("---")
        if st.button("🔓 Log Out"): logout()

    if selected == "Chat": chat_page()
    elif selected == "Readability": readability_page()
    elif selected == "Admin": admin_page()
else:
    if st.session_state['page'] == 'login': login_page()
    elif st.session_state['page'] == 'register': register_page()
    elif st.session_state['page'] == 'forgot': forgot_page()


Writing app.py


In [ ]:
import os

print("EMAIL:", os.getenv("EMAIL_ADDRESS"))
print("PASSWORD:", os.getenv("EMAIL_PASSWORD"))

EMAIL: None
PASSWORD: None


In [ ]:
import os
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

# Kill old sessions
ngrok.kill()
!pkill -f streamlit

# Load secrets
email_password = userdata.get("EMAIL_PASSWORD")
email_address = userdata.get("EMAIL_ADDRESS")
ngrok_token = userdata.get("NGROK_AUTHTOKEN")

os.environ["EMAIL_PASSWORD"] = email_password
os.environ["EMAIL_ADDRESS"] = email_address
os.environ["JWT_SECRET"] = "super-secret-change-me"

# Authenticate ngrok
ngrok.set_auth_token(ngrok_token)

# Start Streamlit
env = os.environ.copy()

process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port=8501"],
    env=env
)

time.sleep(5)

# Start ngrok tunnel
public_url = ngrok.connect(8501)

print("🚀 App Running at:", public_url)
print("Press ENTER to stop the app...")

# Wait for user input
input()

# Stop everything
process.terminate()
ngrok.kill()

print("✅ App stopped. You can run other files now.")

🚀 App Running at: NgrokTunnel: "https://kenda-burliest-vividly.ngrok-free.dev" -> "http://localhost:8501"
Press ENTER to stop the app...
